In [60]:
import pandas as pd

In [61]:
from google.colab import files

In [62]:
uploaded = files.upload()

Saving data.csv to data (1).csv


In [63]:
df=pd.read_csv("data.csv")

In [64]:
df.head()

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral


In [65]:
df.shape

(5842, 2)

In [66]:
df.isnull().sum()

,0
Sentence,0
Sentiment,0


In [67]:
df.duplicated().sum()

np.int64(6)

In [68]:
df.drop_duplicates(inplace=True)

In [69]:
df.shape

(5836, 2)

In [70]:
import re

def clean_data(text):
  text=re.sub(r"/r/n"," ",text)
  text=re.sub(r"http\S+"," ",text)
  text=re.sub(r"<.*?>"," ",text)
  text=re.sub(r"/s+"," ",text)
  return text

In [71]:
df["Sentence"]=df["Sentence"].apply(clean_data)
df["Sentiment"]=df["Sentiment"].apply(clean_data)

In [72]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

df["Sentiment"] = df["Sentiment"].map(label2id)

In [73]:
df.head()

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,2
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",0
2,"For the last quarter of 2010 , Componenta 's n...",2
3,According to the Finnish-Russian Chamber of Co...,1
4,The Swedish buyout firm has sold its remaining...,1


In [74]:
from sklearn.model_selection import train_test_split
x=df['Sentence']
y=df['Sentiment']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

Loading and Training the model (FinBERT)

In [75]:
from transformers import AutoTokenizer

In [76]:
tokenizer=AutoTokenizer.from_pretrained("ProsusAI/finbert")

In [77]:
train_encodings=tokenizer(
    x_train.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings=tokenizer(
    x_test.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

In [78]:
import torch
from torch.utils.data import Dataset
class FinancialNewsDataset(Dataset):
  def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

  def __len__(self):
        return len(self.labels)

  def __getitem__(self, idx):
    item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    item["labels"] = torch.tensor(self.labels[idx])
    return item

In [79]:
train_dataset = FinancialNewsDataset(train_encodings, y_train)
test_dataset = FinancialNewsDataset(test_encodings, y_test)

In [80]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [81]:
from transformers import AutoModelForSequenceClassification

In [82]:
model=AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [83]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [84]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [85]:
from torch.optim import AdamW
optimizer=AdamW(
    model.parameters(),
    lr=2e-5
)

Training the model

In [86]:
from tqdm import tqdm
epochs = 5
model.train()
for epoch in range(epochs):
  total_loss=0
  for batch in tqdm(train_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    optimizer.zero_grad()
    outputs = model(
      input_ids=input_ids,
      attention_mask=attention_mask,
      labels=labels
      )
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

100%|██████████| 292/292 [01:42<00:00,  2.84it/s]


Epoch 1 Loss: 0.6202


100%|██████████| 292/292 [01:42<00:00,  2.86it/s]


Epoch 2 Loss: 0.3125


100%|██████████| 292/292 [01:41<00:00,  2.86it/s]


Epoch 3 Loss: 0.2158


100%|██████████| 292/292 [01:42<00:00,  2.86it/s]


Epoch 4 Loss: 0.1821


100%|██████████| 292/292 [01:41<00:00,  2.87it/s]

Epoch 5 Loss: 0.1603


In [87]:
model.save_pretrained("finbert_sentiment_model")
tokenizer.save_pretrained("finbert_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('finbert_sentiment_model/tokenizer_config.json',
 'finbert_sentiment_model/tokenizer.json')

In [88]:
# for downloading the paramters to device from colab
!zip -r finbert_sentiment_model.zip finbert_sentiment_model
files.download("finbert_sentiment_model.zip")

updating: finbert_sentiment_model/ (stored 0%)
updating: finbert_sentiment_model/config.json (deflated 54%)
updating: finbert_sentiment_model/model.safetensors (deflated 7%)
updating: finbert_sentiment_model/tokenizer_config.json (deflated 43%)
updating: finbert_sentiment_model/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Testing

In [89]:
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
  for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

In [90]:
from sklearn.metrics import accuracy_score

accuracy=accuracy_score(all_labels, all_predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7860


In [92]:
from sklearn.metrics import classification_report

print(classification_report(
    all_labels,
    all_predictions,
    target_names=[
        "Negative",
        "Neutral",
        "Positive"
    ]
))

              precision    recall  f1-score   support

    Negative       0.46      0.54      0.49       172
     Neutral       0.83      0.83      0.83       625
    Positive       0.89      0.82      0.86       371

    accuracy                           0.79      1168
   macro avg       0.73      0.73      0.73      1168
weighted avg       0.80      0.79      0.79      1168

